# SmolDocling-256M-preview — DIMER E2E handwritten-line transcription adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/smoldocling-document-extraction-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/smoldocling-document-extraction-pipeline/blob/main/tutorials/smoldocling_document_extraction_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-docling--project%2FSmolDocling--256M--preview-ffcc4d?style=flat)](https://huggingface.co/docling-project/SmolDocling-256M-preview) [![Upstream](https://img.shields.io/badge/Upstream-docling--project%2Fdocling-181717?style=flat&logo=github&logoColor=white)](https://github.com/docling-project/docling) [![arXiv](https://img.shields.io/badge/arXiv-2503.11576-b31b1b.svg)](https://arxiv.org/abs/2503.11576)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** document page image → DocTags markup under one of the seven supported instructions, and bounded supervised fine-tuning of one instruction — `Convert this page to docling.` on transcribed text lines, through the DocTags output — using the pinned SmolDocling-256M-preview weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/smoldocling_document_extraction_pipeline/`, at revision `cf2deb2675d6`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8` (~518 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `docling-project/SmolDocling-256M-preview` snapshot (a 513 MB `model.safetensors`; no pickle is opened anywhere), fetches the first eight row groups of the Belfort-line test shard from the Hugging Face Hub at an immutable revision with HTTPS range requests (about 44 MB; each row group refused on any SHA-256 or byte-total mismatch), validates the 800 line records and splits them by line into 600 / 60 / 140, converts a synthetic report page through the inference contract with an input manifest and a rejection probe, scores the frozen model's `Convert this page to docling.` over the 140 held-out lines on the text its DocTags carry (character and word error rates) beside an empty-string and a constant-transcript baseline, runs a bounded fine-tuning of the last eight decoder layers on cached prefix hidden states with a DocTags line target and validation-CER epoch selection, scores the held-out lines again, re-runs six held-out lines and the report page with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify transcript parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a Tesla T4 the default path took about 25 minutes of cell time (eight epochs 1084 s, frozen scoring of 140 lines 157 s); a CUDA runtime is used automatically when present, and **a CPU runtime is not practical for the default path** (greedy decoding of some 800 lines plus 600 cached forwards of a 256M-parameter model).

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip of line images plus a `transcripts.csv` (`file`, `text`, optional `id`; one row per image, at least eight images). The records pass through the same validation, image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the Belfort sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

SmolDocling-256M-preview is an Idefics3-arrangement vision–language model built for document conversion: a SigLIP vision encoder turns each 512-px tile of the page (the longest edge resized to 2048 px, aspect ratio preserved, plus one global view) into 64 visual tokens through a pixel-shuffle connector, and a 30-layer SmolLM2 decoder reads them with one of seven supported instructions and generates **DocTags** — a markup in which every element (`<section_header_level_1>`, `<text>`, `<otsl>` table, `<picture>`, …) is preceded by four `<loc_N>` tokens on a 0–500 grid (256,484,928 parameters in all, published under the **CDLA-Permissive-2.0** licence). Decoding is **greedy** (`do_sample=False`), deterministic on a fixed device and dtype. The output is **generated markup with no score**: a well-formed document is not evidence that its words are right.

What this notebook adds to inference is **adaptation of one instruction on transcribed lines, through the DocTags output**. The instruction stays `Convert this page to docling.`; the lines are nineteenth-century French council minutes in cursive — the Belfort-line dataset — far outside the rendered-document distribution the preview was trained on, and on them the frozen model emits a `<text>` element (or a `<picture>`) whose content is empty, spaced-out letters or a loop: a character error rate of **1.439** on the 140 held-out lines (the build record's Tesla T4 figure). So the honest question is narrow: does a bounded fine-tuning of the last eight decoder layers on 600 transcribed lines — trained to emit exactly the DocTags the model already uses for one text element that fills the image, with the transcript inside it — move the held-out **CER** and **WER** on a line-disjoint test split past two **non-adapted baselines** and the frozen model, and what does it do to the report page the same decoder converts? Three sibling rows adapted GOT-OCR 2.0, Florence-2 and SmolVLM-500M on the same split; this is the fourth model on the same 800 lines, and the only one whose output contract is a document markup. Nothing here is a claim about your documents or your script: it is one seeded split of one small labelled set.

**Snapshot note:** the pinned revision ships `model.safetensors` (a 13-file manifest with the tokenizer, chat template and processor files) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the processor or the model is constructed. The pipeline loads the checkpoint in **float32 on every device** (the inference-only tutorial used bfloat16 on CUDA): the adapter is trained in float32 and overlays without a cast, and CPU, Tesla-class and consumer GPUs then run the same arithmetic.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled line set with its transcripts, validate it and split it by line without leakage; convert a synthetic report page through the public API and read DocTags correctly (elements, location tokens, OTSL cells, the `truncated` flag, a `sample-sanity` report against words you rendered yourself); measure the frozen instruction's corpus CER and WER on the text its DocTags carry beside two non-adapted baselines; run a bounded fine-tuning with the model's own instruction-tuning loss and a DocTags line target, explicit hyperparameters and validation-based epoch selection; evaluate on a line-disjoint test split; look at the adapted transcripts next to the frozen ones and the references, and at what the report page's conversion does after the shared decoder was tuned; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** conversion of PDFs or multi-page documents (one page image per call), export to Markdown/HTML/JSON documents (that is `docling_core`, not installed here), sampling or beam search, instructions other than the seven the upstream README lists, fine-tuning of the vision encoder, the connector, the embeddings, the output head or the first 22 decoder layers, fine-tuning of any instruction but `Convert this page to docling.`, a metric for layout or table structure (the report page's element counts stay sanity evidence), evaluation on an OCR or document-conversion benchmark proper (only one seeded 800-line sample is scored here), and any claim that French cursive minutes stand in for your documents. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime with a CUDA GPU (Google Colab or Kaggle GPU, Python 3.12). The default path uses CUDA automatically when present. Generation is batched for the corpus stages — the prompt length varies with the image's tile count, so batches are left-padded — and the build record measured 157 s to score 140 lines and 1084 s for the eight epochs (caching the prefix hidden states for 600 lines took 337 s) on a Tesla T4, about 25 minutes of cell time for the whole path; a CPU runtime would take hours. The pinned `torch==2.14.0` install and the 513 MB checkpoint are the large downloads of the run; the row groups are about 44 MB.
- **Knowledge:** basic Python and PIL; what a chat template, a user turn and greedy decoding are; what DocTags elements and location tokens are; what character and word error rate measure and why they are not capped at 1; why a self-rendered page is a plumbing check while a held-out split of one labelled set is a measurement of that set only.
- **Data contract:** records are `{id, image, text}` — `image` a PIL image (or a file decodable by Pillow) with sides within 16..16,384 px and at most 4096² pixels, `text` its transcript (1..512 characters after whitespace runs are collapsed; case and punctuation kept). Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 8..5,000 records; splitting de-duplicates by decoded pixels so no image lands in two splits. BYOD accepts one zip (or directory) of images plus a `transcripts.csv` in the layout named above.
- **Validation is structural, not semantic:** every image is decoded and every transcript checked for length, but nothing checks that a transcript says what its image shows — a mislabelled set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads eight row groups of `default/test/0000.parquet` from `https://huggingface.co/datasets/Teklia/Belfort-line/resolve/<revision>/` at the immutable parquet-conversion revision `c4a74bbd…` with HTTPS range requests (the parquet footer plus about 44 MB of row-group bytes out of a 210 MB shard), each row group pinned by SHA-256 and byte total in the carried `samples.py` and refused on any mismatch. Belfort-line is published under the MIT licence (Teklia; Tarride et al. 2023); nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `docling-project/SmolDocling-256M-preview` snapshot (~518 MB in total) at revision `ce51f56c4ebe…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'smoldocling-document-extraction-pipeline',
    'repository_revision': 'cf2deb2675d6626657f0599a067897c3049e833c',
    'embedded_module': 'src/smoldocling_document_extraction_pipeline/pipeline.py',
    'embedded_modules': ['src/smoldocling_document_extraction_pipeline/pipeline.py', 'src/smoldocling_document_extraction_pipeline/metrics.py', 'src/smoldocling_document_extraction_pipeline/samples.py'],
    'module_sha256': '862367cd66629a844aa01fa67ddc56e1a689004f17ca444d5df6fdbfaecfea03',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/smoldocling_document_extraction_pipeline/` @ `cf2deb2675d6`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/smoldocling_document_extraction_pipeline/pipeline.py`

In [ ]:
"""Document-to-DocTags conversion with the pinned ``docling-project/SmolDocling-256M-preview`` checkpoint, plus the
adaptation contract for one instruction: corpus evaluation of the text a DocTags conversion carries against
transcribed lines, bounded fine-tuning of the last decoder layers on cached prefix hidden states with a DocTags line
target, and a verified adapter artifact.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Idefics3 architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

import hashlib
import json
import random
import re
import time
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "docling-project/SmolDocling-256M-preview"
MODEL_REVISION = "ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8"
MODEL_LICENSE = "cdla-permissive-2.0"
MODEL_KEY = "smoldocling-256m-preview"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The instructions the pinned README's "Supported Instructions" table lists. Any other instruction
# is refused: the model was trained on these forms and a paraphrase is undefined behaviour.
INSTRUCTIONS = (
    "Convert this page to docling.",
    "Convert chart to table.",
    "Convert formula to LaTeX.",
    "Convert code to text.",
    "Convert table to OTSL.",
    "Find all 'text' elements on the page, retrieve all section headers.",
    "Detect footer elements on the page.",
)
DEFAULT_INSTRUCTION = INSTRUCTIONS[0]
# Generation ceilings. 8192 is the max_new_tokens the pinned README's transformers example passes and
# the text model's max_position_embeddings (config.json); the default is a practical page budget.
MAX_NEW_TOKENS = 8192
DEFAULT_MAX_NEW_TOKENS = 2048
DECODING = "greedy"
# Input ceilings. The processor resizes so the longest edge is 2048 px and splits the page into
# 512-px tiles of 64 visual tokens each plus one global view (preprocessor_config.json), so image
# cost is bounded; the side and area ceilings only guard memory during decoding and resizing (a
# 9,000 px wide text line is accepted).
MAX_IMAGE_SIDE = 16_384
MAX_IMAGE_PIXELS = 4096 * 4096
MIN_IMAGE_SIDE = 16
WEIGHTS_FILE = "model.safetensors"
END_OF_UTTERANCE = "<end_of_utterance>"  # the assistant turn's terminator in the snapshot chat template

# Adaptation contract: the last `TRAINABLE_LAYERS` of the 30 SmolLM2 decoder layers and the final norm are the
# adapter; the SigLIP vision encoder, the connector, the embeddings, the output head and the earlier decoder
# layers stay frozen, so their output for every training line is computed once and cached. The target of one
# transcribed line is the DocTags the model itself emits for a single text element that fills the image.
PARAMETER_COUNT = 256_484_928
DECODER_LAYERS = 30
TRAINABLE_LAYERS = 8
ADAPTER_PARAMETERS = 28_321_344
DEFAULT_LINE_MAX_NEW_TOKENS = 160  # the corpus stages' budget per text line (the DocTags wrapper is 25 tokens)
LINE_DOCTAGS = "<doctag><text><loc_0><loc_0><loc_500><loc_500>{text}</text>\n</doctag>"
_TRAINABLE_FIRST_LAYER = DECODER_LAYERS - TRAINABLE_LAYERS
_TRAINABLE_PREFIXES = tuple(f"model.text_model.layers.{i}." for i in range(_TRAINABLE_FIRST_LAYER, DECODER_LAYERS)) + ("model.text_model.norm.",)
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = 1
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 5_000
EVAL_BATCH_SIZE = 8
CACHE_BATCH_SIZE = 4
GRAD_CLIP = 1.0
# Tokens the decoder emits around the answer; stripped from the returned DocTags (the README example
# decodes with skip_special_tokens=False so the DocTags markup survives, then removes these).
_TERMINATORS = ("<end_of_utterance>", "<|im_end|>")
# DocTags element tags counted by doctags_summary (added_tokens.json names them; the loc grid is
# 0..500 per axis, four <loc_N> tokens per element box).
_ELEMENT_TAGS = (
    "section_header_level_1",
    "section_header_level_2",
    "section_header_level_3",
    "text",
    "paragraph",
    "list_item",
    "ordered_list",
    "unordered_list",
    "otsl",
    "picture",
    "caption",
    "formula",
    "code",
    "page_header",
    "page_footer",
    "footnote",
    "chart",
    "key_value_region",
)
_TAG_RE = re.compile(r"</?([a-z_]+(?:_[0-9]+)?)>")
_LOC_RE = re.compile(r"<loc_[0-9]+>")
_OTSL_CELL_RE = re.compile(r"<(?:fcel|ecel|ched|rhed|srow|lcel|ucel|xcel|nl)>")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _weight_digest(root: Path) -> str | None:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        return None
    with open(manifest_path, encoding="utf-8") as handle:
        entries = json.load(handle).get("files", [])
    return next((e["sha256"] for e in entries if e["path"] == WEIGHTS_FILE), None)


def normalise_text(text: str) -> str:
    """The transcript form the corpus measures use: whitespace runs collapsed to one space, ends stripped."""
    return " ".join(str(text).split())


def edit_distance(reference: Sequence[Any], hypothesis: Sequence[Any]) -> int:
    """Levenshtein distance (insertions + deletions + substitutions, unit cost) between two sequences."""
    previous = list(range(len(hypothesis) + 1))
    for row_index, ref_item in enumerate(reference, 1):
        current = [row_index]
        for column_index, hyp_item in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[column_index] + 1, previous[column_index - 1] + (ref_item != hyp_item)))
        previous = current
    return previous[-1]


def line_doctags(text: str) -> str:
    """The DocTags a transcribed line is trained to produce: one `<text>` element whose box fills the image."""
    return LINE_DOCTAGS.format(text=normalise_text(text))


def _trainable_names(model: Any) -> list[str]:
    """The last `TRAINABLE_LAYERS` text-decoder layers and the final norm; the vision encoder, the connector, the
    embeddings, the output head and the earlier decoder layers stay frozen."""
    return [name for name, _ in model.named_parameters() if name.startswith(_TRAINABLE_PREFIXES)]


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}")
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith(_TRAINABLE_PREFIXES) for n in names):
        raise ValueError(f"artifact tensors must all belong to the last {TRAINABLE_LAYERS} decoder layers or the final norm")
    adapter = manifest.get("adapter") or {}
    if adapter.get("instruction") not in INSTRUCTIONS:
        raise ValueError("artifact manifest must record adapter.instruction, one of the supported INSTRUCTIONS")


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def build_messages(instruction: str) -> list[dict[str, Any]]:
    """One user turn: an image placeholder then the instruction, in the snapshot chat-template shape."""
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]


def doctags_to_text(doctags: str) -> str:
    """Plain text carried by a DocTags string: tags and <loc_N> tokens removed, whitespace collapsed.

    OTSL table cells become space-separated words; structure is lost. This is the text a caller would
    compare with an OCR reference, not a document export (use docling_core for that).
    """
    text = _LOC_RE.sub(" ", doctags)
    text = _OTSL_CELL_RE.sub(" ", text)
    text = _TAG_RE.sub(" ", text)
    return " ".join(text.split())


def doctags_summary(doctags: str) -> dict[str, Any]:
    """Count the DocTags elements in a conversion: which structure the model claims the page has.

    Counts opening tags per element type, the number of <loc_N> tokens (four per located element),
    whether the string is wrapped in <doctag>…</doctag>, and how many OTSL table cells appear.
    """
    opened = Counter(
        match.group(1) for match in _TAG_RE.finditer(doctags) if not match.group(0).startswith("</")
    )
    counts = {tag: opened.get(tag, 0) for tag in _ELEMENT_TAGS}
    return {
        "counts": counts,
        "n_elements": sum(counts.values()),
        "n_loc_tokens": len(_LOC_RE.findall(doctags)),
        "n_table_cells": len(_OTSL_CELL_RE.findall(doctags)),
        "wrapped_in_doctag": doctags.lstrip().startswith("<doctag>")
        and doctags.rstrip().endswith("</doctag>"),
        "n_chars": len(doctags),
    }


def _tokens(text: str) -> list[str]:
    return text.lower().split()


def word_error_rate(reference: str, hypothesis: str) -> float:
    """Word error rate of ``hypothesis`` against ``reference`` after lower-casing and whitespace tokenisation.

    Levenshtein edits over words divided by reference words; punctuation is **not** stripped, so a
    stray comma counts. The metric a caller would use to score ``doctags_to_text`` against a known page.
    """
    ref, hyp = _tokens(reference), _tokens(hypothesis)
    if not ref:
        raise ValueError("reference must contain at least one word")
    previous = list(range(len(hyp) + 1))
    for row_index, ref_token in enumerate(ref, 1):
        current = [row_index]
        for column_index, hyp_token in enumerate(hyp, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[column_index] + 1,
                    previous[column_index - 1] + (ref_token != hyp_token),
                )
            )
        previous = current
    return previous[-1] / len(ref)


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    if width * height > MAX_IMAGE_PIXELS:
        raise ValueError(f"image area {width * height} px > MAX_IMAGE_PIXELS {MAX_IMAGE_PIXELS}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one page image as PIL.Image.Image (any mode, converted to RGB) plus one supported instruction",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "image_max_pixels": MAX_IMAGE_PIXELS,
    "instructions": list(INSTRUCTIONS),
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False), deterministic on a fixed device and dtype",
    "preprocessing": (
        "image converted to RGB; the processor resizes so the longest edge is 2048 px (aspect ratio "
        "preserved) and splits it into 512-px tiles of 64 visual tokens each plus one global view; the "
        "instruction is wrapped in the snapshot's chat template as one user turn (see build_messages)"
    ),
    "output": "DocTags markup (docling_core-compatible), decoded without dropping the tag tokens",
}


def _check_inputs(image: Any, instruction: Any, max_new_tokens: Any) -> tuple[Image.Image, str, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``convert`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if not isinstance(instruction, str):
        raise TypeError("instruction must be a str")
    if instruction not in INSTRUCTIONS:
        raise ValueError(f"instruction {instruction!r} is not one of the supported INSTRUCTIONS")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, instruction, max_new_tokens


def validate_inputs(
    image: Image.Image,
    *,
    instruction: str = DEFAULT_INSTRUCTION,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``convert`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked_instruction, checked_tokens = _check_inputs(image, instruction, max_new_tokens)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (convert takes one page image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "instruction": checked_instruction,
        "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_text: str | None = None,
    expected_counts: Mapping[str, int] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_text`` (the words the page really carries) the report carries ``word_error_rate``
    of ``doctags_to_text`` against it; with ``expected_counts`` (element tag -> expected number) it
    carries one ``element_count`` entry per tag comparing expected and observed. Either makes the
    verdict ``sample-sanity``; without both it is ``not-measurable`` and the report says what labelled
    data would make the task measurable.
    """
    doctags = str(result["doctags"])
    summary = doctags_summary(doctags)
    base = {
        "task": "document page image -> DocTags (layout, reading order, OCR, tables)",
        "score_semantics": (
            "generated markup carries no score, no probability and no correctness signal; well-formed "
            "tags are not evidence that the text or layout is right. Greedy decoding makes the output "
            "reproducible on a fixed device and dtype, which is a reproducibility property, not a quality one"
        ),
        "instruction": result.get("instruction"),
        "sample_kind": sample_kind,
        "doctags_summary": summary,
        "truncated": result.get("truncated"),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    metrics: list[dict[str, Any]] = []
    if reference_text:
        metrics.append(
            {
                "id": "word_error_rate",
                "value": word_error_rate(reference_text, doctags_to_text(doctags)),
                "normalisation": (
                    "lower-cased, whitespace-tokenised, tags and <loc_N> removed; punctuation kept"
                ),
                "estimation": "one page, no dispersion estimate",
            }
        )
    for tag, expected in (expected_counts or {}).items():
        if tag not in _ELEMENT_TAGS:
            raise ValueError(f"unknown element tag {tag!r}; expected one of {_ELEMENT_TAGS}")
        metrics.append(
            {
                "id": "element_count",
                "tag": tag,
                "expected": int(expected),
                "observed": summary["counts"][tag],
                "estimation": "one page, structural sanity only",
            }
        )
    if not metrics:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference text or expected element counts were supplied for the evaluated page",
            "needs": (
                "pages with ground-truth text and layout (for example DocLayNet-style annotations or the "
                "publisher's source) scored with word_error_rate on the OCR and with layout/table metrics "
                "such as TEDS on the structure; no such labelled set ships with this repository"
            ),
        }
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} sanity measure(s) on one tutorial page whose text and layout you rendered "
            "yourself; plumbing evidence, not a document-conversion benchmark"
        ),
        "needs": (
            "a labelled page set from the deployment domain (scans, publishers, layouts, tables) for any "
            "OCR accuracy, layout or table-structure claim"
        ),
    }


@dataclass
class SmolDoclingPipeline:
    """``_runner(image, instruction, max_new_tokens)`` returns ``{"doctags": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"
    _batch_runner: Callable[..., list[dict[str, Any]]] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> SmolDoclingPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        # float32 on every device: the adapter is trained in float32 and overlays without a cast, and CPU,
        # Tesla-class and consumer GPUs then run the same arithmetic.
        dtype = torch.float32
        processor = AutoProcessor.from_pretrained(location, **common)
        model = AutoModelForImageTextToText.from_pretrained(location, dtype=dtype, **common)
        model = model.eval().to(resolved_device)
        for param in model.parameters():
            param.requires_grad_(False)
        end_id = processor.tokenizer.convert_tokens_to_ids(END_OF_UTTERANCE)
        pad_id = processor.tokenizer.pad_token_id

        def runner(image: Image.Image, instruction: str, max_new_tokens: int) -> dict[str, Any]:
            text = processor.apply_chat_template(build_messages(instruction), add_generation_prompt=True)
            inputs = processor(text=text, images=[image], return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            new_ids = generated[0, inputs["input_ids"].shape[1] :]
            # skip_special_tokens=False keeps the DocTags markup (the tags are added tokens); the
            # terminator tokens are removed afterwards, as the pinned README's example does.
            decoded = processor.batch_decode(new_ids.unsqueeze(0), skip_special_tokens=False)[0]
            return {"doctags": decoded, "new_tokens": int(new_ids.shape[0])}

        def batch_runner(images: Sequence[Image.Image], instruction: str, max_new_tokens: int) -> list[dict[str, Any]]:
            text = processor.apply_chat_template(build_messages(instruction), add_generation_prompt=True)
            processor.tokenizer.padding_side = "left"  # prompts differ in tile count; generation needs left padding
            inputs = processor(text=[text] * len(images), images=[[image] for image in images], return_tensors="pt", padding=True).to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            prompt_len = int(inputs["input_ids"].shape[1])
            out = []
            for row in generated[:, prompt_len:]:
                ids = row.tolist()
                n_new = len(ids)
                for position, token in enumerate(ids):
                    if token in (end_id, pad_id):
                        n_new = position + (token == end_id)
                        break
                decoded = processor.tokenizer.decode(ids[:n_new], skip_special_tokens=False)
                out.append({"doctags": decoded, "new_tokens": int(n_new)})
            return out

        return cls(runner, resolved_device, str(dtype).removeprefix("torch."), source, batch_runner, model, processor, _weight_digest(root))

    def convert(
        self,
        image: Image.Image,
        *,
        instruction: str = DEFAULT_INSTRUCTION,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Run one supported instruction over one page image; ``doctags`` is the decoded markup."""
        rgb, checked_instruction, checked_tokens = _check_inputs(image, instruction, max_new_tokens)
        raw = self._runner(rgb, checked_instruction, checked_tokens)
        if not isinstance(raw, dict) or "doctags" not in raw:
            raise RuntimeError("runner must return a dict with 'doctags'")
        doctags = str(raw["doctags"])
        for terminator in _TERMINATORS:
            doctags = doctags.replace(terminator, "")
        doctags = doctags.strip()
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "doctags": doctags,
            "text": doctags_to_text(doctags),
            "instruction": checked_instruction,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ------------------------------------------------------------------------------------------------------
    # Adaptation contract (one instruction on transcribed text lines, through the DocTags output)
    # ------------------------------------------------------------------------------------------------------

    @staticmethod
    def _strip(doctags: str) -> str:
        for terminator in _TERMINATORS:
            doctags = doctags.replace(terminator, "")
        return doctags.strip()

    def transcribe(
        self,
        images: Sequence[Image.Image],
        *,
        instruction: str = DEFAULT_INSTRUCTION,
        max_new_tokens: int = DEFAULT_LINE_MAX_NEW_TOKENS,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> list[dict[str, Any]]:
        """Convert many images under `instruction` with greedy decoding, in batches (left-padded, since the tile count
        and hence the prompt length vary with the image); one ``{doctags, text, new_tokens, truncated}`` per image, in
        order, where ``text`` is the plain text the DocTags carry (`doctags_to_text`, whitespace-normalised). With an
        injected runner and no batch runner the images are converted one by one through the runner."""
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        checked = [_check_inputs(image, instruction, max_new_tokens)[0] for image in images]
        out: list[dict[str, Any]] = []
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            if self._batch_runner is not None:
                raw = self._batch_runner(batch, instruction, max_new_tokens)
            else:
                raw = [self._runner(image, instruction, max_new_tokens) for image in batch]
            if not isinstance(raw, list) or len(raw) != len(batch) or any(not isinstance(r, dict) or "doctags" not in r for r in raw):
                raise RuntimeError("batch runner must return one dict with 'doctags' per image")
            for item in raw:
                doctags = self._strip(str(item["doctags"]))
                new_tokens = int(item.get("new_tokens", 0))
                out.append({"doctags": doctags, "text": normalise_text(doctags_to_text(doctags)), "new_tokens": new_tokens, "truncated": new_tokens >= max_new_tokens})
            if progress is not None:
                progress(len(out), len(checked))
        return out

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained for adapt/save_artifact/load_artifact")
        return self._model, self._processor

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        instruction: str = DEFAULT_INSTRUCTION,
        max_new_tokens: int = DEFAULT_LINE_MAX_NEW_TOKENS,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> dict[str, Any]:
        """Convert every validated record under `instruction` and score the text the DocTags carry as a transcript
        with ``metrics.ocr_metrics`` (micro and macro CER / WER, exact match). Works with an injected runner too."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import ocr_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        items = self.transcribe([r["image"] for r in checked], instruction=instruction, max_new_tokens=max_new_tokens, batch_size=batch_size, progress=progress)
        hypotheses = [item["text"] for item in items]
        metrics = ocr_metrics(hypotheses, checked)
        metrics.update(
            {
                "hypotheses": hypotheses,
                "doctags": [item["doctags"] for item in items],
                "instruction": instruction,
                "truncated": sum(item["truncated"] for item in items),
                "new_tokens": sum(item["new_tokens"] for item in items),
                "max_new_tokens": max_new_tokens,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None,
        *,
        instruction: str = DEFAULT_INSTRUCTION,
        epochs: int = 8,
        lr: float = 1e-4,
        batch_size: int = 8,
        seed: int = 0,
        max_new_tokens: int = DEFAULT_LINE_MAX_NEW_TOKENS,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the last `TRAINABLE_LAYERS` text-decoder layers and the final norm on transcribed lines
        with the causal language-model loss over the assistant turn — the DocTags of one `<text>` element that fills
        the image, carrying the transcript (`line_doctags`), then the end-of-utterance token; the image tokens and the
        user turn are masked — the checkpoint's own instruction-tuning objective and output contract. The frozen
        prefix — vision encoder, connector, embeddings and the first decoder layers — is run once per line under no
        gradient and its output hidden states are cached, so each step runs only the trainable tail; the loss equals
        the full model's loss exactly. AdamW (no weight decay), gradient clipping at `GRAD_CLIP`, seeded shuffling, no
        scheduler, no augmentation. Epoch 0 records the frozen model's validation metrics; the epoch with the lowest
        validation CER is kept (the final one without a validation split). On any exception the frozen weights are
        restored."""
        model, processor = self._require_model()  # refuse before importing torch
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        _check_inputs(Image.new("RGB", (16, 16)), instruction, max_new_tokens)  # the instruction contract
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        names = _trainable_names(model)
        name_set = set(names)
        device = torch.device(self.device)
        language_model = model.model.text_model
        first = _TRAINABLE_FIRST_LAYER
        tokenizer = processor.tokenizer
        end_id = tokenizer.convert_tokens_to_ids(END_OF_UTTERANCE)
        pad_id = tokenizer.pad_token_id
        chat_text = processor.apply_chat_template(build_messages(instruction), add_generation_prompt=True)
        frozen_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        previous_adapter = self.adapter
        cudnn_flags = (torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark)
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False  # repeatable on one device
        history: list[dict[str, Any]] = []
        started = time.perf_counter()

        def _val() -> dict[str, Any] | None:
            if val_checked is None:
                return None
            result = self.evaluate(val_checked, instruction=instruction, max_new_tokens=max_new_tokens)
            return {"cer": result["cer"], "wer": result["wer"], "cer_macro": result["cer_macro"], "exact_match": result["exact_match"], "n": result["n"]}

        def _encode(batch: Sequence[Mapping[str, Any]]) -> tuple[dict[str, Any], Any]:
            processor.tokenizer.padding_side = "right"  # the cache is right-padded: causal tokens never see a pad
            encoded = processor(text=[chat_text] * len(batch), images=[[r["image"]] for r in batch], return_tensors="pt", padding=True)
            ids, labels_list = [], []
            for i, record in enumerate(batch):
                prompt_ids = encoded["input_ids"][i][encoded["attention_mask"][i].bool()]
                target = torch.tensor(tokenizer(line_doctags(record["text"]), add_special_tokens=False)["input_ids"] + [end_id], dtype=torch.long)
                ids.append(torch.cat([prompt_ids, target]))
                labels_list.append(torch.cat([torch.full_like(prompt_ids, -100), target]))
            length = max(int(x.shape[0]) for x in ids)
            input_ids = torch.full((len(batch), length), pad_id, dtype=torch.long)
            labels = torch.full((len(batch), length), -100, dtype=torch.long)
            mask = torch.zeros((len(batch), length), dtype=torch.long)
            for i, (x, y) in enumerate(zip(ids, labels_list, strict=True)):
                input_ids[i, : x.shape[0]] = x
                labels[i, : y.shape[0]] = y
                mask[i, : x.shape[0]] = 1
            inputs = {"input_ids": input_ids.to(device), "attention_mask": mask.to(device), "pixel_values": encoded["pixel_values"].to(device)}
            if encoded.get("pixel_attention_mask") is not None:
                inputs["pixel_attention_mask"] = encoded["pixel_attention_mask"].to(device)
            return inputs, labels

        def _tail_loss(hidden: Any, labels: Any) -> Any:
            position_ids = torch.arange(hidden.shape[1], device=device).unsqueeze(0).expand(hidden.shape[0], -1)
            embeddings = language_model.rotary_emb(hidden, position_ids)
            for layer in language_model.layers[first:]:
                hidden = layer(hidden, attention_mask=None, position_ids=position_ids, position_embeddings=embeddings)
            # logits only where a target token is predicted: the same cross-entropy as the full model's, without a
            # batch x length x vocabulary logit tensor
            targets = labels[:, 1:]
            keep = targets != -100
            logits = model.lm_head(language_model.norm(hidden[:, :-1][keep]))
            return torch.nn.functional.cross_entropy(logits.float(), targets[keep])

        try:
            # 1. cache the frozen prefix: the hidden states entering the first trainable layer, per line
            cache: list[tuple[Any, Any]] = []
            for start in range(0, len(train_checked), CACHE_BATCH_SIZE):
                batch = train_checked[start : start + CACHE_BATCH_SIZE]
                inputs, labels = _encode(batch)
                with torch.no_grad():
                    hidden = model.model(**inputs, output_hidden_states=True).hidden_states[first]
                for k in range(len(batch)):
                    n = int(inputs["attention_mask"][k].sum())
                    cache.append((hidden[k, :n].detach().to("cpu"), labels[k, :n]))
                del hidden
            cache_seconds = round(time.perf_counter() - started, 3)
            # 2. train the tail on the cached states
            params = []
            for name, param in model.named_parameters():
                if name in name_set:
                    param.requires_grad_(True)
                    params.append(param)
            n_trainable = sum(p.numel() for p in params)
            entry = {"epoch": 0, "train_loss": None, "val": _val(), "note": "frozen model"}
            history.append(entry)
            if progress is not None:
                progress(entry)
            best_epoch, best_score = 0, (history[0]["val"] or {}).get("cer", float("inf"))
            best_state = frozen_state
            optimizer = torch.optim.AdamW(params, lr=float(lr), weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            width = int(cache[0][0].shape[1])
            for epoch in range(1, epochs + 1):
                model.train()
                order = list(range(len(cache)))
                rng.shuffle(order)
                losses = []
                for start in range(0, len(order), batch_size):
                    items = [cache[k] for k in order[start : start + batch_size]]
                    length = max(int(h.shape[0]) for h, _ in items)
                    hidden = torch.zeros((len(items), length, width), dtype=items[0][0].dtype)
                    labels = torch.full((len(items), length), -100, dtype=torch.long)
                    for k, (h, lab) in enumerate(items):
                        hidden[k, : h.shape[0]] = h
                        labels[k, : lab.shape[0]] = lab
                    loss = _tail_loss(hidden.to(device), labels.to(device))
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                    optimizer.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": _val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked is None or entry["val"]["cer"] < best_score:
                    best_epoch, best_score = epoch, (entry["val"] or {}).get("cer", float("inf"))
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            model.load_state_dict(best_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
        except BaseException:
            model.load_state_dict(frozen_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
            self.adapter = previous_adapter
            raise
        finally:
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags
        self.adapter = {
            "instruction": instruction,
            "target": LINE_DOCTAGS,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "first_trainable_layer": first,
            "epochs": epochs,
            "batch_size": batch_size,
            "best_epoch": best_epoch,
            "selection": "lowest validation CER" if val_checked is not None else "final epoch (no validation split)",
            "loss": "causal language-model cross-entropy over the assistant turn (the DocTags line target and the end-of-utterance token); image tokens and the user turn masked; computed on the cached frozen-prefix hidden states",
            "lr": float(lr),
            "seed": seed,
            "max_new_tokens": max_new_tokens,
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "cache_seconds": cache_seconds,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained tensors as safetensors plus a manifest naming the base, the digests, the instruction and
        the training configuration. Requires a prior `adapt`."""
        model, _processor = self._require_model()  # refuse before importing torch
        import torch
        from safetensors.torch import save_file

        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHTS_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the last decoder layers and the final norm."""
        model, _processor = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model)
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> SmolDoclingPipeline:
        """Check the adapter manifest against the base snapshot's recorded weight digest, load the verified base, then
        overlay the adapter (checked again, and the tensor set, before deserialising). A refused manifest never loads
        a model."""
        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        _check_artifact_manifest(manifest, artifact, _weight_digest(root) or "")
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/smoldocling_document_extraction_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level text-recognition measures and two non-neural baselines, in plain Python.

``ocr_metrics`` scores one hypothesis string per record against ``record['text']``: the character error rate and the
word error rate as **micro** averages (total Levenshtein edits over total reference characters or words — the
corpus CER/WER of the handwriting-recognition literature) and as **macro** averages (mean per-line rate), the exact-
match rate, and the counts behind them. Neither rate is capped: a hypothesis longer than its reference can push a
rate above 1.0, which is the signal that the model is generating text the image does not carry. The baselines
answer without looking at the image — the empty string, or one constant training transcript — and are scored by
the same function.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import edit_distance, normalise_text` removed — names are kernel globals defined by the carried modules

METRIC_DEFINITIONS = {
    "cer": "character error rate, micro: total character edits (insertions + deletions + substitutions) over total reference characters; 0 is perfect, values above 1 mean over-generation",
    "wer": "word error rate, micro: total word edits over total reference words after lower-casing and whitespace tokenisation; punctuation kept",
    "cer_macro": "mean of the per-line character error rates (each line weighted equally regardless of length)",
    "wer_macro": "mean of the per-line word error rates",
    "exact_match": "fraction of lines whose hypothesis equals the reference after whitespace normalisation",
}
MEDOID_POOL = 120  # training transcripts considered when choosing the constant baseline (quadratic cost)


def _words(text: str) -> list[str]:
    return text.lower().split()


def ocr_metrics(hypotheses: Sequence[str], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Score one hypothesis per record against `record['text']`; per-line rows plus the corpus rates. Raises when
    the lengths differ or nothing is scored."""
    if len(hypotheses) != len(records) or not records:
        raise ValueError("hypotheses and records must be non-empty and the same length")
    rows = []
    char_edits = word_edits = ref_chars = ref_words = hyp_chars = exact = 0
    for hypothesis, record in zip(hypotheses, records, strict=True):
        reference = normalise_text(record["text"])
        hyp = normalise_text(str(hypothesis))
        if not reference:
            raise ValueError(f"record {record.get('id')!r} has an empty reference")
        ce = edit_distance(reference, hyp)
        we = edit_distance(_words(reference), _words(hyp))
        n_words = len(_words(reference))
        rows.append({"id": record.get("id"), "ref_chars": len(reference), "hyp_chars": len(hyp), "char_edits": ce, "cer": ce / len(reference), "word_edits": we, "wer": we / n_words, "exact": hyp == reference})
        char_edits += ce
        word_edits += we
        ref_chars += len(reference)
        ref_words += n_words
        hyp_chars += len(hyp)
        exact += hyp == reference
    n = len(rows)
    return {
        "n": n,
        "cer": char_edits / ref_chars,
        "wer": word_edits / ref_words,
        "cer_macro": sum(r["cer"] for r in rows) / n,
        "wer_macro": sum(r["wer"] for r in rows) / n,
        "exact_match": exact / n,
        "char_edits": char_edits,
        "ref_chars": ref_chars,
        "hyp_chars": hyp_chars,
        "word_edits": word_edits,
        "ref_words": ref_words,
        "rows": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def empty_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Predict the empty string for every line: CER and WER are exactly 1.0 by construction (every reference
    character is a deletion). The floor any recogniser must beat to be doing better than silence."""
    result = ocr_metrics([""] * len(records), records)
    result["baseline"] = "empty string for every line"
    return result


def medoid_transcript(train: Sequence[Mapping[str, Any]], *, pool: int = MEDOID_POOL) -> str:
    """The training transcript (among the first `pool`) with the smallest summed character error rate to the others:
    the single constant string that best matches the corpus on average."""
    texts = [normalise_text(r["text"]) for r in train[:pool]]
    texts = [t for t in texts if t]
    if not texts:
        raise ValueError("no non-empty training transcripts")
    return min(texts, key=lambda candidate: sum(edit_distance(other, candidate) / len(other) for other in texts))


def constant_baseline(train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]], *, pool: int = MEDOID_POOL) -> dict[str, Any]:
    """Predict one constant training transcript (the medoid) for every line: what corpus statistics alone buy
    without reading the image — usually a CER near 1.0, since edits are dominated by substitutions."""
    text = medoid_transcript(train, pool=pool)
    result = ocr_metrics([text] * len(records), records)
    result["baseline"] = f"constant training transcript {text!r}"
    result["transcript"] = text
    return result

**Module 3/3:** `src/smoldocling_document_extraction_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled text-line datasets for the adaptation contract: the digest-pinned Belfort sample, the record contract
and its structural validation, image-disjoint splitting, and the BYOD loader.

A record is ``{id, image, text}`` where ``image`` is a PIL image of one text line (or any region whose transcript is
known; sides within the pipeline's ceilings) and ``text`` its transcript — the string the model is asked to
generate for that image. Whitespace runs in ``text`` are collapsed; case and punctuation are kept.

The default sample is drawn from the Belfort-line dataset (Teklia; the minutes of the Belfort municipal council,
19th–20th century French handwriting transcribed by a crowdsourcing campaign; Tarride et al. 2023, **MIT**) as
converted to parquet by the Hugging Face Hub at an immutable revision: the first ``CORPUS_ROW_GROUPS`` row groups of
the test shard are read with HTTPS range requests (about 5.5 MB each; the shard's declared size is checked first and
every row group's decoded content is refused unless its SHA-256 matches the pin). Every line image is 128 px tall;
widths run from about 145 to 9,000 px. The domain gap to the model's printed-text training distribution is the point
of the sample: the frozen model reads almost none of it.
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MODEL_ID, normalise_text, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Belfort-line (test split), first eight parquet row groups"
CORPUS_REPO = "Teklia/Belfort-line"
CORPUS_REVISION = "c4a74bbd39f2df314752e7e6026649a39d365cbb"  # refs/convert/parquet commit on the Hub
CORPUS_FILE = "default/test/0000.parquet"
CORPUS_BYTES = 210_579_166
CORPUS_ROWS = 3_819
CORPUS_ROW_GROUPS = 8  # of 39; 100 lines each
CORPUS_LICENSE = "MIT (Teklia; Belfort municipal council minutes, Zenodo record 8041668; Tarride et al. 2023, https://doi.org/10.1145/3604951.3605517)"
CORPUS_LANGUAGE = "fr"
CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"
# SHA-256 over the concatenated image bytes + UTF-8 transcript of each row group, in row order, and that byte total.
ROW_GROUP_PINS: dict[int, tuple[str, int]] = {
    0: ("1dc3141e4809ea628b17c3ca7b81d64e6ca92bce18dd5765ecd618bfc7867954", 5_481_145),
    1: ("c9d3b52013933c803f4886edbce68da0ae483347a4a6ff3e0f5ced1db4a7e653", 5_465_901),
    2: ("6e0578a90a07a9e25e65b765881d3fa33d6a797624425e01026980d7287f0bf6", 5_379_166),
    3: ("00cdfb7aabe924f31b9f1bb1ba4849040051e5619567b68bf99fdcbcab15131a", 5_821_303),
    4: ("fc063442fb20e7a60c2533ab44dcc69a22ad59f5ce24921fe6af5f53ceab7e1a", 5_163_559),
    5: ("2d7e29331bd4e93e0c8a1caa9a83b8f1ede9b17af6dae9377b83f56c56f05689", 4_713_140),
    6: ("49423d91780cb184c4b0069630e85acccb314a113256ced9692a5138c4782ef1", 5_069_794),
    7: ("3a010831456f185399579b4ecf9d46222f95368c3cdbbc2ff103a258b9c16e2f", 5_309_891),
}
DEFAULT_CACHE_DIR = Path("weights") / "belfort"

SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 600, "validation": 60, "test": 140}  # of the 800 lines the eight row groups hold
SAMPLE_DIGEST = "b7e1dd684691a0eedb63a609311f4964e7732e5c1a8d254fe4e1293a8cd0964d"  # dataset_digest over the three default splits together; tests pin it
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MIN_TEXT_CHARS = 1
MAX_TEXT_CHARS = 512
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to read a
    parquet footer and a few row groups without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}", "User-Agent": "smoldocling-document-extraction-pipeline"})
        with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _declared_size(url: str) -> int:
    request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "smoldocling-document-extraction-pipeline"})
    with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 (pinned https URL)
        length = response.headers.get("Content-Length")
    if length is None:
        raise ValueError(f"{url}: no Content-Length in the HEAD response")
    return int(length)


def _group_digest(rows: Sequence[Mapping[str, Any]]) -> tuple[str, int]:
    digest, total = hashlib.sha256(), 0
    for row in rows:
        data = row["image"]["bytes"]
        text = str(row["text"]).encode("utf-8")
        digest.update(data)
        digest.update(text)
        total += len(data) + len(text)
    return digest.hexdigest(), total


def fetch_corpus(
    *, cache_dir: str | Path | None = None, groups: Sequence[int] | None = None, opener: Any = None
) -> dict[int, list[dict[str, Any]]]:
    """Return the pinned row groups as lists of `{image, text}` (JPEG bytes, transcript), from the cache (one parquet
    file per row group) or the Hub (footer + the row groups it needs, over range requests). Every row group's decoded
    content is refused unless its SHA-256 and byte total match `ROW_GROUP_PINS`; a fresh fetch also checks the shard's
    declared size and row count."""
    import pyarrow.parquet as pq

    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    wanted = list(groups) if groups is not None else sorted(ROW_GROUP_PINS)
    out: dict[int, list[dict[str, Any]]] = {}
    reader = None
    for group in wanted:
        if group not in ROW_GROUP_PINS:
            raise ValueError(f"row group {group} has no pin; pinned groups are {sorted(ROW_GROUP_PINS)}")
        local = cache / f"test-rg{group}.parquet"
        rows: list[dict[str, Any]] | None = None
        if local.is_file():
            rows = pq.read_table(local).to_pylist()
            if _group_digest(rows) != ROW_GROUP_PINS[group]:
                rows = None  # stale or corrupt cache: refetch
        if rows is None:
            if reader is None:
                if opener is not None:
                    reader = pq.ParquetFile(opener(CORPUS_URL))
                else:
                    declared = _declared_size(CORPUS_URL)
                    if declared != CORPUS_BYTES:
                        raise ValueError(f"{CORPUS_FILE}: declared size {declared} != pinned {CORPUS_BYTES}")
                    reader = pq.ParquetFile(_HttpRangeFile(CORPUS_URL, CORPUS_BYTES))
                if reader.metadata.num_rows != CORPUS_ROWS:
                    raise ValueError(f"{CORPUS_FILE}: {reader.metadata.num_rows} rows, pinned {CORPUS_ROWS}")
            table = reader.read_row_group(group, columns=["image", "text"])
            rows = table.to_pylist()
            digest, total = _group_digest(rows)
            if (digest, total) != ROW_GROUP_PINS[group]:
                raise ValueError(f"{CORPUS_FILE} row group {group}: sha256 {digest} / {total} bytes != pinned {ROW_GROUP_PINS[group]}")
            pq.write_table(table, local)
        out[group] = [{"image": r["image"]["bytes"], "text": str(r["text"])} for r in rows]
    return out


def read_corpus(groups: Mapping[int, Sequence[Mapping[str, Any]]]) -> list[dict[str, Any]]:
    """Decode the verified row groups into records (one per line; lines with an empty transcript are skipped)."""
    out = []
    for group in sorted(groups):
        for index, row in enumerate(groups[group]):
            text = normalise_text(row["text"])
            if not text:
                continue
            image = Image.open(io.BytesIO(row["image"]))
            image.load()
            out.append({"id": f"belfort-test-{group * 100 + index}", "image": image.convert("RGB"), "text": text, "source_row_group": group})
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """Seeded line-level draw: shuffle the records and cut `sizes` (train / validation / test) in order."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    pool = [dict(r) for r in records]
    random.Random(seed).shuffle(pool)
    needed = sum(sizes.values())
    if len(pool) < needed:
        raise ValueError(f"only {len(pool)} records available, need {needed}")
    out, cursor = {}, 0
    for name, count in sizes.items():
        out[name] = pool[cursor : cursor + count]
        cursor += count
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)


# ---------------------------------------------------------------------------------------------------------
# Record contract
# ---------------------------------------------------------------------------------------------------------


def _open(image: Any, where: str) -> Image.Image:
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{where}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{where}: must be a PIL.Image.Image or a file path")
    return image


def _check_record(record: Any, index: int) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/image/text")
    for key in ("id", "image", "text"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    try:
        image = validate_image(_open(record["image"], f"{where}.image"))
    except TypeError as exc:
        raise ValueError(f"{where}: {exc}") from exc
    except ValueError as exc:
        raise ValueError(f"{where}: {exc}") from exc
    if not isinstance(record["text"], str):
        raise ValueError(f"{where}: text must be a str")
    text = normalise_text(record["text"])
    if not MIN_TEXT_CHARS <= len(text) <= MAX_TEXT_CHARS:
        raise ValueError(f"{where}: text has {len(text)} characters after whitespace normalisation; {MIN_TEXT_CHARS}..{MAX_TEXT_CHARS} are required")
    item = {"id": rid, "image": image, "text": text}
    if "source_row_group" in record:
        item["source_row_group"] = record["source_row_group"]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a text-line dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, text} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked, ids = [], set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    chars = [len(r["text"]) for r in checked]
    words = [len(r["text"].split()) for r in checked]
    widths = [r["image"].width for r in checked]
    heights = [r["image"].height for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "text_chars": {"min": min(chars), "max": max(chars), "total": sum(chars)},
        "text_words": {"total": sum(words)},
        "image_width": {"min": min(widths), "max": max(widths)},
        "image_height": {"min": min(heights), "max": max(heights)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size-prefixed) — the identity a split is made disjoint on."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over (id, image digest, normalised text)."""
    parts = sorted(f"{r['id']}:{image_digest(r['image'])}:{normalise_text(r['text'])}" for r in records)
    return _sha256_bytes("\n".join(parts).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n = len(unique)
    n_test = max(1, round(n * test_fraction))
    n_val = round(n * val_fraction)
    if n - n_test - n_val < 1:
        raise ValueError(f"{n} distinct images are too few to split into train/validation/test")
    return {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a directory or zip holding line images and a `transcripts.csv` with the columns `file` and `text`
    (and optionally `id`); every image file must have a transcript row and every row an image."""
    source = Path(path)
    members: dict[str, bytes] = {}
    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file():
                members[file.name] = file.read_bytes()
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                if not info.is_dir():
                    members[Path(info.filename).name] = archive.read(info)  # flattened; no extractall
    else:
        raise ValueError(f"{source} is neither a directory nor a zip file")
    if "transcripts.csv" not in members:
        raise ValueError("BYOD data must include transcripts.csv with the columns file and text")
    rows = list(csv.DictReader(io.StringIO(members["transcripts.csv"].decode("utf-8-sig"))))
    if not rows or "file" not in rows[0] or "text" not in rows[0]:
        raise ValueError("transcripts.csv must have the columns file and text")
    out = []
    for row in rows:
        name = Path(str(row.get("file", "")).strip()).name
        if name not in members:
            raise ValueError(f"transcripts.csv names a missing image: {name}")
        try:
            image = Image.open(io.BytesIO(members[name]))
            image.load()
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"BYOD file is not a decodable image: {name}") from exc
        rid = str(row.get("id", "") or "").strip()
        out.append({"id": rid or re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "image": image.convert("RGB"), "text": str(row.get("text", ""))})
    listed = {Path(str(r.get("file", "")).strip()).name for r in rows}
    unlisted = [n for n in members if n != "transcripts.csv" and n not in listed]
    if unlisted:
        raise ValueError(f"{len(unlisted)} image file(s) have no transcripts.csv row, e.g. {unlisted[0]}")
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A summary table (id, image size, transcript length, transcript, provenance) in the BYOD `transcripts.csv`
    column layout plus extras (`file` names the id; the images themselves are not written)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "file", "width", "height", "chars", "words", "text", "source_row_group"])
        for r in records:
            text = normalise_text(r["text"])
            writer.writerow([r["id"], f"{r['id']}.jpg", r["image"].width, r["image"].height, len(text), len(text.split()), text, r.get("source_row_group", "")])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `13`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `ce51f56c4ebe…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `SmolDoclingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "smoldocling-256m-preview",
  "modelId": "docling-project/SmolDocling-256M-preview",
  "revision": "ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8",
  "files": [
    {
      "path": "README.md",
      "bytes": 16108,
      "sha256": "9b82c4dd1b38340656da55d628d63bf319c5fc4700811148687b6d8070e7e493"
    },
    {
      "path": "added_tokens.json",
      "bytes": 3667,
      "sha256": "fc79a032b551636ad0fe6c0e16bfe38c43b5843895cbb0544a4a5919818472cc"
    },
    {
      "path": "chat_template.json",
      "bytes": 430,
      "sha256": "b585e3598909a5687f9f9d738d35223724dedef256b9b274e1cbfb32b13c74bf"
    },
    {
      "path": "config.json",
      "bytes": 3903,
      "sha256": "57af2810c65b9896a8d1d65c67aabdb9296d497aa10a6329f4bb2ddce623586f"
    },
    {
      "path": "generation_config.json",
      "bytes": 141,
      "sha256": "0758109c85e7f7d6b0202ebf643bb07c5625b3363e389854b414d9a701becc28"
    },
    {
      "path": "merges.txt",
      "bytes": 466391,
      "sha256": "0b54e8aa4e53d5383e2e4bc635a56b43f9647f7b13832d5d9ecd8f82dac4f510"
    },
    {
      "path": "model.safetensors",
      "bytes": 513028808,
      "sha256": "cdcdf5d823c5684029c7d8e52177cf10f9034b3aba6577549cfb1a9ce36ad0a2"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 486,
      "sha256": "6cb6e36d6fcb88ca1502c4a26750715dc3e7dedddc9a8f17b27d8d167d1457e7"
    },
    {
      "path": "processor_config.json",
      "bytes": 68,
      "sha256": "e7bff42da73ae9eec9042ef20e066e11f1ee20f025358ff79131e3c0fb549b46"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 1069,
      "sha256": "aa0ff906077086dfa9734a7f97f68c825877a48f9468807be65504495cdeef09"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3547443,
      "sha256": "7c5cf6233a3dc8b9e54fb729ee6e771bdf5f0d65fd9075b5e60bed837959deee"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 27362,
      "sha256": "b38f39506a4fa7d2604015a6c303df320675cfeb7ba1f5969e1c44032727107b"
    },
    {
      "path": "vocab.json",
      "bytes": 800662,
      "sha256": "82b84012e3add4d01d12ba14442026e49b8cbbaead1f79ecf3d919784f82dc79"
    }
  ],
  "totalBytes": 517896538
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = SmolDoclingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Belfort lines, the transcripts and the split

`fetch_corpus` returns the eight pinned row groups from the cache under `weights/belfort/` or the Hub at the pinned parquet-conversion revision — `pyarrow` reads the shard's footer and exactly those row groups over HTTPS range requests; every cached file is re-hashed and every fetched row group refused on any SHA-256 or byte-total mismatch — and `read_corpus` turns each row into a record: the line image (128 px tall, 145 to 8,956 px wide) and its crowdsourced transcript with whitespace runs collapsed. `build_sample_dataset` draws a seeded line-level split (600 / 60 / 140). `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no image (by decoded-pixel digest) is shared, and the training split's summary table is written to `outputs/smoldocling_document_extraction_train.csv`.

Look for: 800 lines and 33,117 reference characters, three digests, and four refusal probes — a duplicate id, an empty transcript, an image above the side ceiling, and a dataset too small to use — each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    t0 = time.perf_counter()
    corpus_groups = fetch_corpus(cache_dir='weights/belfort')
    corpus = read_corpus(corpus_groups)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} @ {CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'row_groups': len(corpus_groups), 'lines': sum(len(v) for v in corpus_groups.values()), 'bytes': sum(len(r['image']) + len(r['text'].encode('utf-8')) for v in corpus_groups.values() for r in v), 'seconds': round(time.perf_counter() - t0, 1)}
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(train_records, 'outputs/smoldocling_document_extraction_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'chars': manifest['text_chars'], 'words': manifest['text_words']['total'], 'width': manifest['image_width'], 'height': manifest['image_height'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
example['image'].save('outputs/smoldocling_document_extraction_example_line.png')
print({'example': {'id': example['id'], 'image': list(example['image'].size), 'text': example['text']}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty transcript': [{**train_records[0], 'text': '   '}, *train_records[1:8]],
    'image above the side ceiling': [{**train_records[0], 'image': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 32))}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Convert a synthetic report page through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: an 850 × 1100 report page rendered in code — a heading, two paragraphs, an 8×5 and a 5×3 ruled table and a closing sentence — whose words and element counts are known because you rendered them; a different image family from the handwritten lines, and a page the adapted model will convert again in Section 9. `validate_inputs` applies exactly the checks `convert` applies (a PIL image with sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` and at most `MAX_IMAGE_PIXELS`, one of the seven `INSTRUCTIONS`, `max_new_tokens` in 1..`MAX_NEW_TOKENS`) and returns an input manifest; a paraphrased instruction is validated too and its rejection recorded as a finding. The instruction and the token budget are **caller-owned request parameters**. `convert` returns the DocTags, the text they carry (`doctags_to_text`), `new_tokens` and a `truncated` flag that is true when the budget was exhausted — **no score exists** in the markup. `doctags_summary` counts the elements the model claims the page has, and `evaluation_report` with the rendered words and counts is `sample-sanity`: a `word_error_rate` and one `element_count` entry per tag against references you drew, plumbing evidence for one page (the inference-only card recorded 0.19 with both tables cell-perfect); whether the model *reads handwriting* is what Section 6 measures on 140 lines with a metric.

In [ ]:
def synthetic_page(width=850, height=1100):
    """Heading, two paragraphs, an 8x5 and a 5x3 ruled table, one closing line. Returns page, reference text, expected counts."""
    page = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(page)
    body, head = ImageFont.load_default(size=15), ImageFont.load_default(size=22)
    words = 'quarterly revenue by region and product line for the fiscal year with notes on methodology'.split()
    reference = ['Annual Report: Regional Results']
    d.text((70, 50), reference[0], fill='black', font=head)
    y = 95
    for _para in range(2):
        for line in range(6):
            text = ' '.join(words[(line * 3 + k) % len(words)] for k in range(11 - (line % 3)))
            d.text((70, y), text, fill=(40, 40, 40), font=body)
            reference.append(text)
            y += 20
        y += 16
    for x0, y0, x1, y1, rows, cols, first in ((70, 330, 780, 660, 8, 5, 'Region'), (70, 760, 430, 990, 5, 3, 'Item')):
        d.rectangle([x0, y0, x1, y1], outline='black', width=2)
        rh, cw = (y1 - y0) / rows, (x1 - x0) / cols
        d.line([(x0, y0 + rh), (x1, y0 + rh)], fill='black', width=2)
        for r in range(2, rows):
            d.line([(x0, y0 + rh * r), (x1, y0 + rh * r)], fill=(120, 120, 120), width=1)
        for c in range(1, cols):
            d.line([(x0 + cw * c, y0), (x0 + cw * c, y1)], fill=(120, 120, 120), width=1)
        for r in range(rows):
            for c in range(cols):
                token = (first if c == 0 else f'Q{c}') if r == 0 else (f'North {r}' if c == 0 else f'{(r * 7 + c * 13) % 97 + 1},{(r * 31 + c) % 900 + 100:03d}')
                d.text((x0 + cw * c + 8, y0 + rh * r + rh / 2 - 8), token, fill='black', font=body)
                reference.append(token)
    tail = 'Table 2 summarises the line items; see the appendix for the full breakdown.'
    d.text((70, 1010), tail, fill=(40, 40, 40), font=body)
    reference.append(tail)
    return page, ' '.join(reference), {'section_header_level_1': 1, 'text': 3, 'otsl': 2}


page, page_reference, page_counts = synthetic_page()
page_name = 'synthetic_report_page_850x1100'
page_sha256 = hashlib.sha256(np.asarray(page).tobytes()).hexdigest()
PAGE_MAX_NEW_TOKENS = 2048
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_IMAGE_PIXELS': MAX_IMAGE_PIXELS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DEFAULT_LINE_MAX_NEW_TOKENS': DEFAULT_LINE_MAX_NEW_TOKENS, 'DECODING': DECODING, 'INSTRUCTIONS': list(INSTRUCTIONS), 'LINE_DOCTAGS': LINE_DOCTAGS, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'device': pipe.device, 'dtype': pipe.dtype}})
input_manifest = validate_inputs(page, instruction=DEFAULT_INSTRUCTION, max_new_tokens=PAGE_MAX_NEW_TOKENS, names=[page_name])
try:
    validate_inputs(page, instruction='Please convert this page to markdown.')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'unsupported-instruction-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/smoldocling_document_extraction_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'page': page_name, 'sha256': page_sha256[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def convert_page(pipeline, label):
    started = time.perf_counter()
    result = pipeline.convert(page, instruction=DEFAULT_INSTRUCTION, max_new_tokens=PAGE_MAX_NEW_TOKENS)
    seconds = round(time.perf_counter() - started, 3)
    summary = doctags_summary(result['doctags'])
    checks = {
        'doctags_is_text': isinstance(result['doctags'], str) and bool(result['doctags']),
        'greedy_settings': result['generation']['do_sample'] is False and result['generation']['decoding'] == DECODING,
        'budget_honoured': result['new_tokens'] <= PAGE_MAX_NEW_TOKENS,
        'identity_reported': result['model_id'] == MODEL_ID and result['model_revision'] == MODEL_REVISION,
    }
    if not all(checks.values()):
        raise RuntimeError(f'convert output failed a sanity check: {checks}')
    report = evaluation_report(result, page_reference, page_counts, sample_kind='synthetic (rendered in this notebook)')
    with open(f'outputs/smoldocling_document_extraction_page_{label}.json', 'w', encoding='utf-8') as handle:
        json.dump({'result': result, 'summary': summary, 'report': report}, handle, indent=2, ensure_ascii=False)
    wer = next((m['value'] for m in report['metrics'] if m['id'] == 'word_error_rate'), None)
    counts = {m['tag']: (m['expected'], m['observed']) for m in report['metrics'] if m['id'] == 'element_count'}
    print({label: {'seconds': seconds, 'checks': checks, 'new_tokens': result['new_tokens'], 'truncated': result['truncated'], 'n_elements': summary['n_elements'], 'n_table_cells': summary['n_table_cells'], 'word_error_rate': None if wer is None else round(wer, 3), 'element_counts (expected, observed)': counts, 'verdict': report['verdict']}})
    print(result['text'][:300] + ('…' if len(result['text']) > 300 else ''))
    return result, seconds, checks, report


frozen_page, frozen_page_seconds, frozen_page_checks, frozen_page_report = convert_page(pipe, 'frozen')

## 6. Baselines and the frozen model on the test lines

Two non-adapted baselines frame the adaptation, each scored by `ocr_metrics` (carried in `metrics.py`): the **character error rate** and **word error rate** as micro averages — total Levenshtein edits over total reference characters or words, the corpus CER/WER of the handwriting-recognition literature — beside the macro (per-line mean) rates and the exact-match rate. Neither rate is capped: a hypothesis longer than its reference pushes the rate **above 1.0**, the signal that the model is generating text the line does not carry. The **empty-string** baseline predicts nothing and scores CER 1.0 exactly (every reference character is a deletion) — the floor any recogniser must beat to do better than silence. The **constant-transcript** baseline predicts one training transcript — the medoid, the line closest on average to the others — for every test line: what corpus statistics buy without reading the image. The **frozen model** is scored by `pipe.evaluate`, which converts every line under `DEFAULT_INSTRUCTION` in left-padded batches of `EVAL_BATCH_SIZE` under a `LINE_MAX_NEW_TOKENS` budget, strips the DocTags with `doctags_to_text` and scores the text that remains as the hypothesis. Expect the frozen model **near the empty baseline**: the build record measured 1.439 (hypotheses 0.78 times the reference length — the frozen model answers the transcription instruction with almost nothing — 62 of the 140 hypotheses are empty and 100 are three characters or fewer — while 21 lines loop on a digit or a syllable to the 160-token budget (29 truncated), which is how a CER above 1.0 coexists with hypotheses shorter than the references); read four of them under the references, with the DocTags they came from.

In [ ]:
METRICS = ('cer', 'wer', 'cer_macro', 'exact_match')
LINE_MAX_NEW_TOKENS = 160  # @param {type:"integer"}

baseline_empty = empty_baseline(test_records)
baseline_constant = constant_baseline(train_records, test_records)
print({'empty_baseline': {k: round(baseline_empty[k], 3) for k in METRICS}, 'n': baseline_empty['n'], 'note': baseline_empty['baseline']})
print({'constant_baseline': {k: round(baseline_constant[k], 3) for k in METRICS}, 'note': baseline_constant['baseline']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, instruction=DEFAULT_INSTRUCTION, max_new_tokens=LINE_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE)
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'n': frozen_test['n'], 'ref_chars': frozen_test['ref_chars'], 'hyp_chars': frozen_test['hyp_chars'], 'truncated': frozen_test['truncated'], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
for record, hypothesis, doctags in zip(test_records[:4], frozen_test['hypotheses'][:4], frozen_test['doctags'][:4], strict=True):
    print({'id': record['id'], 'reference': record['text'], 'frozen': hypothesis[:120], 'doctags': doctags[:90]})

## 7. Bounded fine-tuning of the last decoder layers

`pipe.adapt` trains only the last eight of the 30 SmolLM2 decoder layers and the final norm — 28,321,344 of 256,484,928 parameters — while the SigLIP vision encoder, the connector, the embeddings, the output head and the first 22 decoder layers stay frozen. Each training line is the chat-templated user turn (the tiles' visual tokens and the instruction) followed by the assistant turn: the **DocTags line target** — `<doctag><text><loc_0><loc_0><loc_500><loc_500>` + transcript + `</text></doctag>`, exactly the markup the frozen model already emits for one text element that fills the image — and the end-of-utterance token; the loss is the **causal language-model cross-entropy** over the assistant turn with the user turn masked out — the checkpoint's own instruction-tuning objective and output contract. Because everything before layer 22 is frozen, its output for every training line is computed once under no gradient and cached (the **frozen-prefix cache**), and each step runs only the eight trainable layers, the norm and the output head on those cached states — the loss equals the full model's loss exactly, at a fraction of the cost. AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, seeded shuffling, no scheduler, no augmentation. Epoch 0 records the frozen model's validation rates; every epoch is scored on the 60 validation lines, and the epoch with the **lowest validation CER** is kept.

Watch the validation CER fall from 1.600 to 0.871 (epoch 8 in the build record) while the loss drops from about 1.36 to 0.04: the adapted model gets 22 % of the characters right and 2 of 140 lines exact. The three sibling rows on the same split — GOT-OCR 2.0 (0.759 CER), Florence-2 (0.797) and SmolVLM-500M (0.890) — are compared in the card.

In [ ]:
EPOCHS = 8  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in METRICS})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, instruction=DEFAULT_INSTRUCTION, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, max_new_tokens=LINE_MAX_NEW_TOKENS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'instruction': adapt_result['instruction'], 'target': adapt_result['target'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'first_trainable_layer': adapt_result['first_trainable_layer'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'loss': adapt_result['loss'], 'cache_seconds': adapt_result['cache_seconds'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test lines were never used for training or epoch selection, and no image appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6 and the four systems are put side by side. Read it in this order: **CER** first (the measure the epoch was selected on — the build record measured 1.439 → **0.782**, past both baselines), then **WER** (2.491 → 0.975), then the hypothesis length (from 0.78 times the reference length to 0.85), then the exact-match rate (2 of the 140 lines read perfectly). The cell asserts the adapted CER is below the frozen one and below the empty baseline's 1.0. One hundred and forty lines from one seeded split give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a result on one French council's minutes says nothing about other hands, other languages or other scripts until you measure them.

In [ ]:
adapted_test = pipe.evaluate(test_records, instruction=DEFAULT_INSTRUCTION, max_new_tokens=LINE_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE)
adapted_val = pipe.evaluate(val_records, instruction=DEFAULT_INSTRUCTION, max_new_tokens=LINE_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE)
comparison = {metric: {'empty': round(baseline_empty[metric], 3), 'constant': round(baseline_constant[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
comparison['hypothesis_length'] = {'ref_chars': adapted_test['ref_chars'], 'frozen_hyp_chars': frozen_test['hyp_chars'], 'adapted_hyp_chars': adapted_test['hyp_chars'], 'frozen_truncated': frozen_test['truncated'], 'adapted_truncated': adapted_test['truncated']}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'instruction': DEFAULT_INSTRUCTION,
    'target': LINE_DOCTAGS,
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'empty': {k: v for k, v in baseline_empty.items() if k != 'rows'}, 'constant': {k: v for k, v in baseline_constant.items() if k != 'rows'}},
    'frozen_test': {k: v for k, v in frozen_test.items() if k != 'rows'},
    'validation_metrics': {k: v for k, v in adapted_val.items() if k != 'rows'},
    'test_metrics': {k: v for k, v in adapted_test.items() if k != 'rows'},
    'per_line': [{**frozen_row, 'frozen_hypothesis': frozen_hyp, 'frozen_doctags': frozen_dt, 'adapted_cer': adapted_row['cer'], 'adapted_hypothesis': adapted_hyp, 'adapted_doctags': adapted_dt} for frozen_row, frozen_hyp, frozen_dt, adapted_row, adapted_hyp, adapted_dt in zip(frozen_test['rows'], frozen_test['hypotheses'], frozen_test['doctags'], adapted_test['rows'], adapted_test['hypotheses'], adapted_test['doctags'], strict=True)],
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/smoldocling_document_extraction_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['cer'] < frozen_test['cer']
assert adapted_test['cer'] < baseline_empty['cer']
print({'report': 'outputs/smoldocling_document_extraction_evaluation_report.json', 'adapted_beats_both_baselines': adapted_test['cer'] < min(baseline_empty['cer'], baseline_constant['cer'])})

## 9. Look at the lines, convert the page again, export the adapter and reload it

Six held-out lines are written as panels (`outputs/smoldocling_document_extraction_examples/`: the line image with the reference, the frozen answer and the adapted answer beneath it) so the numbers can be checked by eye: the adapted rows should carry the cursive the frozen rows left empty or spaced out. The report page from Section 5 is then converted again by the adapted model — the decoder that was tuned answers every instruction, so this is a small look at what the adaptation did *outside* its corpus: the build record measured before adaptation 630 new tokens, page WER 0.192, element counts (expected, observed) `section_header_level_1` 1/1, `text` 3/3, `otsl` 2/2, verdict `sample-sanity`; after adaptation 92 new tokens, page WER 0.709, element counts (expected, observed) `section_header_level_1` 1/0, `text` 3/1, `otsl` 2/0, verdict `sample-sanity` — one page of evidence, not a measurement.

`pipe.save_artifact` writes the trained tensors — the eight decoder layers and the norm, about 113 MB in float32 — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the instruction and the line target, the training configuration and the epoch history (OUT8). `SmolDoclingPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the last eight decoder layers and the norm, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical transcripts on eight test lines (VER4).

In [ ]:
import shutil

examples_dir = Path('outputs/smoldocling_document_extraction_examples')
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir(parents=True)
caption_font = ImageFont.load_default(size=18)
for record, frozen_hyp, adapted_hyp in zip(test_records[:6], frozen_test['hypotheses'][:6], adapted_test['hypotheses'][:6], strict=True):
    line = record['image']
    width = min(1400, line.width)
    line = line.resize((width, max(1, round(line.height * width / record['image'].width))))
    sheet = Image.new('RGB', (max(width, 1400), line.height + 96), (255, 255, 255))
    sheet.paste(line, (0, 0))
    marker = ImageDraw.Draw(sheet)
    for i, (tag, text) in enumerate((('REF', record['text']), ('FROZEN', frozen_hyp), ('ADAPTED', adapted_hyp))):
        marker.text((8, line.height + 6 + i * 28), f'{tag}: {text[:140]}', fill=(20, 20, 20) if tag != 'FROZEN' else (150, 40, 40), font=caption_font)
    sheet.save(examples_dir / f"{record['id']}.png")
print({'examples': sorted(p.name for p in examples_dir.iterdir()), 'rows': ['reference', 'frozen answer', 'adapted answer']})

adapted_page, adapted_page_seconds, adapted_page_checks, adapted_page_report = convert_page(pipe, 'adapted')

artifact_dir = Path('outputs/smoldocling_document_extraction_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'smoldocling_document_extraction', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'instruction': artifact_manifest['adapter']['instruction'], 'best_epoch': artifact_manifest['adapter']['best_epoch']})

reloaded = SmolDoclingPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [item['text'] for item in pipe.transcribe([r['image'] for r in test_records[:8]], instruction=DEFAULT_INSTRUCTION, max_new_tokens=LINE_MAX_NEW_TOKENS)]
after = [item['text'] for item in reloaded.transcribe([r['image'] for r in test_records[:8]], instruction=DEFAULT_INSTRUCTION, max_new_tokens=LINE_MAX_NEW_TOKENS)]
parity = {'identical_lines': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_lines'] == parity['of']

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'file': CORPUS_FILE, 'license': CORPUS_LICENSE, 'language': CORPUS_LANGUAGE, 'row_groups': sorted(ROW_GROUP_PINS), 'shard_bytes': CORPUS_BYTES},
    'inference_contract': {'input_manifest': input_manifest, 'page': {'name': page_name, 'sha256': page_sha256, 'reference_words': len(page_reference.split()), 'expected_counts': page_counts}, 'frozen': {'result': frozen_page, 'seconds': frozen_page_seconds, 'checks': frozen_page_checks, 'report': frozen_page_report}, 'adapted': {'result': adapted_page, 'seconds': adapted_page_seconds, 'checks': adapted_page_checks, 'report': adapted_page_report}, 'output_files': ['outputs/smoldocling_document_extraction_page_frozen.json', 'outputs/smoldocling_document_extraction_page_adapted.json']},
    'comparison': comparison,
    'examples': 'outputs/smoldocling_document_extraction_examples',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pillow': PIL.__version__, 'device': pipe.device, 'source': pipe.source, 'dtype': pipe.dtype},
}
with open('outputs/smoldocling_document_extraction_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A document-conversion model trained on rendered pages, asked to convert a line of nineteenth-century cursive French, emits the markup it knows and little of the text: the frozen model scores a character error rate of 1.439 on the Belfort lines. A bounded fine-tuning of the last eight decoder layers on 600 transcribed lines — with the transcript inside the model's own `<text>` element — moves it to 0.782 CER and 0.975 WER in the build record (2 of the held-out lines exact), with a 113 MB adapter that reloads line-for-line. That is the claim: the adaptation contract works end to end on one instruction of a document-conversion model with a real labelled set, through the model's own output contract, and the numbers it produces are read as micro and macro rates, against two non-adapted baselines and the frozen model, with the hypothesis length beside them rather than in isolation. Read against the three sibling rows on the same split — GOT-OCR 2.0 at 0.759, Florence-2 at 0.797 and SmolVLM-500M at 0.890 — SmolDocling's 0.782 lands between GOT-OCR and Florence-2 although it is the smallest of the four models (256M) and started from a frozen position that read nothing; the validation CER was still falling at the last epoch (1.024 → 0.871), so eight epochs are a floor for this recipe, not a converged point; 7 of 140 lines end under 0.5 CER and 19 above 1.0; and the cost is visible on the report page — after adaptation the same instruction returns 92 tokens instead of 630, the section header and both tables vanish (element counts 1/0, 3/1, 2/0) and the page WER moves 0.192 → 0.709: the tuned decoder learned lines and page structure paid for it, the drift the card names.

The test split is 140 lines from one seeded draw of one 800-line sample, the validation split that picks the epoch is 60, and both rates are corpus edit distances over one crowdsourced transcription of the text the DocTags carry — not a benchmark, not a measure of layout, reading order or table structure. So a result here says the contract works on one council's minutes, not that the adapted model handles other hands, other languages, other scripts or your scans. The decoder that was tuned answers every instruction: the report page re-converted in Section 9 is one page of evidence about what the tuning did to document conversion (before adaptation 630 new tokens, page WER 0.192, element counts (expected, observed) `section_header_level_1` 1/1, `text` 3/3, `otsl` 2/2, verdict `sample-sanity`; after adaptation 92 new tokens, page WER 0.709, element counts (expected, observed) `section_header_level_1` 1/0, `text` 3/1, `otsl` 2/0, verdict `sample-sanity`), not a measurement, and a deployment that needs page conversion after adapting must measure it. The decoder was adapted, not the vision encoder: what the tiles cannot resolve stays unread.

Three things to carry to real data. **Baselines first:** the empty and constant-transcript rates on *your* transcripts, and the frozen model's hypothesis length, are the numbers to read before any adapted one. **Rates above 1.0:** an uncapped CER tells you the model is generating, not reading; a capped one would hide it. **Leakage:** keep every image in one split (the contract de-duplicates by decoded pixels) and split by page, writer or volume when your lines come from few sources — lines cut from the same page share a hand.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled line set, validate the demonstrated dataset contract without leakage, execute the inference contract for one page and a bounded fine-tuning of one instruction with the model's own objective and output contract, evaluate against two non-adapted baselines and the frozen model on a line-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, transcription quality on any other hand, language or document family, page-conversion quality after adaptation, or production fitness.

**Optional experiments (they do not affect the default path):** raise `EPOCHS` and watch the validation CER pick the epoch; set `LEARNING_RATE` to `5e-5` and read a slower validation curve; lower `LINE_MAX_NEW_TOKENS` to `64` and read how the truncation count changes; change `LINE_DOCTAGS` in the carried module to a `<paragraph>` element and rerun from Section 7 to see whether the element name matters; or bring your own transcribed lines through BYOD and read the two baselines before the adapted number.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/smoldocling-256m-preview/` and rerun Section 3.

## References

- Repository README: https://github.com/kurtvalcorza/smoldocling-document-extraction-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/smoldocling-document-extraction-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/smoldocling-document-extraction-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/docling-project/SmolDocling-256M-preview
- Upstream code: https://github.com/docling-project/docling
- SmolDocling: An ultra-compact vision-language model for end-to-end multi-modal document conversion (Nassar et al., 2025): https://arxiv.org/abs/2503.11576
- Belfort-line dataset (Teklia, MIT): https://huggingface.co/datasets/Teklia/Belfort-line — Tarride et al., Handwritten Text Recognition from Crowdsourced Annotations (HIP 2023): https://doi.org/10.1145/3604951.3605517
- Sibling rows on the same split: https://github.com/kurtvalcorza/got-ocr2-pipeline, https://github.com/kurtvalcorza/florence2-vision-language-pipeline and https://github.com/kurtvalcorza/smolvlm-vision-language-pipeline
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)